# PaleoWave — 08 iDigBio Harvest

Query iDigBio for Nevada Triassic ichthyosaur occurrence records and compare against
existing PBDB dataset. Goal: identify new unique georeferenced localities that could
expand training data, particularly for the under-represented Luning and Gabbs formations.

**iDigBio API:** https://search.idigbio.org/v2/  
**Taxa of interest:** Order Ichthyosauria — families Shastasauridae, Cymbospondylidae,
Mixosauridae, Omphalosauridae, Utatsusauridae, Grippiidae  
**Key genera:** Shonisaurus, Cymbospondylus, Thalattoarchon, Mixosaurus, Omphalosaurus

**Outputs:**
- `data/pbdb/idigbio_raw.csv` — raw iDigBio hits
- `data/pbdb/idigbio_nevada_triassic.csv` — filtered to Nevada + Triassic
- `data/pbdb/paleowave_combined.csv` — PBDB + iDigBio deduplicated
- Summary: new unique localities, coordinate precision assessment

In [1]:
## 1. Imports & Setup
import requests
import pandas as pd
import numpy as np
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = Path('../data')
PBDB_DIR  = DATA_DIR / 'pbdb'
PBDB_DIR.mkdir(exist_ok=True)

IDIGBIO_SEARCH = 'https://search.idigbio.org/v2/search/records'
IDIGBIO_SUMMARY= 'https://search.idigbio.org/v2/summary/count/records'

# Bounding box: Nevada + generous buffer (covers Humboldt Range TRc extent)
# Lat 36-42N, Lon -120 to -114W
NV_BBOX = [-120.0, 36.0, -114.0, 42.0]  # [min_lon, min_lat, max_lon, max_lat]

print('Setup complete.')
print(f'iDigBio search endpoint: {IDIGBIO_SEARCH}')

Setup complete.
iDigBio search endpoint: https://search.idigbio.org/v2/search/records


In [2]:
## 2. Load Existing PBDB Data (for deduplication later)
pbdb = pd.read_csv(PBDB_DIR / 'pbdb_occurrences_clean.csv')
print(f'Existing PBDB records: {len(pbdb)}')
print(f'  Unique localities (rounded to 0.01 deg): '
      f'{pbdb[["longitude","latitude"]].round(2).drop_duplicates().shape[0]}')
print(f'  Formations: {pbdb["formation"].value_counts().to_dict()}')
print(f'  Families: {pbdb["family"].value_counts().to_dict()}')
print()

# Build set of existing coords for dedup
pbdb_coords = set(
    zip(pbdb['longitude'].round(2), pbdb['latitude'].round(2))
)
print(f'PBDB coordinate pairs (0.01 deg): {len(pbdb_coords)}')

Existing PBDB records: 30
  Unique localities (rounded to 0.01 deg): 17
  Formations: {'Prida': 9, 'Favret': 7, 'Luning': 6, 'Gabbs': 1}
  Families: {'Cymbospondylidae': 10, 'Mixosauridae': 5, 'Ichthyosauridae': 4, 'Shastasauridae': 3, 'Omphalosauridae': 2, 'NO_FAMILY_SPECIFIED': 1, 'Utatsusauridae': 1, 'Grippiidae': 1}

PBDB coordinate pairs (0.01 deg): 17


In [3]:
## 3. iDigBio Query Function

def idigbio_search(rq, limit=1000, offset=0):
    """POST to iDigBio — more reliable than GET for complex queries.
    Returns all fields; filter columns post-hoc."""
    body = {
        'rq':     rq,
        'limit':  limit,
        'offset': offset,
    }
    resp = requests.post(
        IDIGBIO_SEARCH,
        json=body,
        headers={'Content-Type': 'application/json'},
        timeout=30
    )
    resp.raise_for_status()
    return resp.json()

def idigbio_count(rq):
    """Count records matching a query."""
    resp = requests.post(
        IDIGBIO_SUMMARY,
        json={'rq': rq},
        headers={'Content-Type': 'application/json'},
        timeout=30
    )
    resp.raise_for_status()
    return resp.json().get('itemCount', 0)

def records_to_df(result):
    """Flatten iDigBio items list to DataFrame.
    Each item has 'data' (raw DwC) and 'indexTerms' (iDigBio parsed).
    We merge both so we get the best of each."""
    rows = []
    for hit in result.get('items', []):
        row = {'uuid': hit.get('uuid', '')}
        row.update(hit.get('indexTerms', {}))   # parsed/normalised fields
        row.update(hit.get('data', {}))          # raw DwC (may override with better values)
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

print('Query functions defined (POST method).')

Query functions defined (POST method).


In [4]:
## 4. Query 1: Order Ichthyosauria — Nevada bounding box

rq_order = {
    'order': 'ichthyosauria',
    'geopoint': {
        'type': 'geo_bounding_box',
        'top_left': {'lat': NV_BBOX[3], 'lon': NV_BBOX[0]},
        'bottom_right': {'lat': NV_BBOX[1], 'lon': NV_BBOX[2]}
    }
}

count_order = idigbio_count(rq_order)
print(f'Ichthyosauria in Nevada bbox: {count_order} records')

if count_order > 0:
    result_order = idigbio_search(rq_order, limit=min(count_order+50, 2000))
    df_order = records_to_df(result_order)
    print(f'  Retrieved: {len(df_order)} records')
    if len(df_order):
        print(f'  Columns: {df_order.columns.tolist()}')
else:
    df_order = pd.DataFrame()
    print('  No records found.')

Ichthyosauria in Nevada bbox: 0 records
  No records found.


In [5]:
## 5. Query 2: Key Genera (no bbox in rq — filter post-hoc)
# Geopoint filter in rq causes 400 when combined with genus on some records.
# Fetch all records for each genus globally, then clip to Nevada bbox.

ICHTHYO_GENERA = [
    'Shonisaurus', 'Cymbospondylus', 'Thalattoarchon',
    'Mixosaurus', 'Omphalosaurus', 'Pessosaurus',
    'Toretocnemus', 'Californosaurus', 'Delphinosaurus'
]

genus_results = []
for genus in ICHTHYO_GENERA:
    rq = {'genus': genus.lower()}   # no geopoint — filter post-hoc
    n = idigbio_count(rq)
    print(f'  {genus:20s}: {n} total records globally')
    if n > 0:
        res = idigbio_search(rq, limit=min(n + 10, 500))
        df_g = records_to_df(res)
        df_g['query_genus'] = genus
        genus_results.append(df_g)
    time.sleep(0.3)

df_genera = pd.concat(genus_results, ignore_index=True) if genus_results else pd.DataFrame()
print(f'\nTotal genus-level records (global): {len(df_genera)}')
print('(Will clip to Nevada bbox in Cell 7)')


  Shonisaurus         : 15 total records globally
  Cymbospondylus      : 90 total records globally
  Thalattoarchon      : 0 total records globally
  Mixosaurus          : 17 total records globally
  Omphalosaurus       : 24 total records globally
  Pessosaurus         : 68 total records globally
  Toretocnemus        : 9 total records globally
  Californosaurus     : 10 total records globally
  Delphinosaurus      : 0 total records globally

Total genus-level records (global): 233
(Will clip to Nevada bbox in Cell 7)


In [6]:
## 6. Query 3: Broad Nevada ichthyosaur — no bbox, filter post-hoc
#  Some records may have imprecise coords that miss bbox; catch via state

rq_state = {
    'order': 'ichthyosauria',
    'stateprovince': 'nevada'
}
count_state = idigbio_count(rq_state)
print(f'Ichthyosauria + stateprovince=nevada: {count_state} records')

if count_state > 0:
    result_state = idigbio_search(rq_state, limit=min(count_state+50, 2000))
    df_state = records_to_df(result_state)
    print(f'  Retrieved: {len(df_state)}')
else:
    df_state = pd.DataFrame()

# Also try without state filter but with formation names
for fm in ['Prida', 'Favret', 'Luning', 'Gabbs']:
    rq_fm = {'formation': fm.lower(), 'order': 'ichthyosauria'}
    n_fm = idigbio_count(rq_fm)
    print(f'  Formation "{fm}": {n_fm} records')

Ichthyosauria + stateprovince=nevada: 1 records
  Retrieved: 1
  Formation "Prida": 0 records
  Formation "Favret": 0 records
  Formation "Luning": 0 records
  Formation "Gabbs": 0 records


In [7]:
## 7. Combine & Deduplicate All iDigBio Results

frames = [df for df in [df_order, df_genera, df_state] if len(df) > 0]

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all.drop_duplicates(subset='uuid')
    print(f'Total unique iDigBio records: {len(df_all)}')
    print(f'Columns present: {sorted(df_all.columns.tolist())}')
    print()

    # iDigBio indexTerms returns coords as a 'geopoint' dict: {'lat': x, 'lon': y}
    # Raw DwC uses 'dwc:decimalLatitude' / 'dwc:decimalLongitude'
    # Handle all three possible column layouts
    def extract_coords(df):
        df = df.copy()
        # Option 1: geopoint dict from indexTerms
        if 'geopoint' in df.columns:
            df['latitude']  = df['geopoint'].apply(
                lambda x: x.get('lat') if isinstance(x, dict) else np.nan)
            df['longitude'] = df['geopoint'].apply(
                lambda x: x.get('lon') if isinstance(x, dict) else np.nan)
        # Option 2: flat decimallatitude/decimallongitude from indexTerms
        elif 'decimallatitude' in df.columns:
            df['latitude']  = pd.to_numeric(df['decimallatitude'], errors='coerce')
            df['longitude'] = pd.to_numeric(df['decimallongitude'], errors='coerce')
        # Option 3: raw DwC camelCase
        elif 'dwc:decimalLatitude' in df.columns:
            df['latitude']  = pd.to_numeric(df['dwc:decimalLatitude'],  errors='coerce')
            df['longitude'] = pd.to_numeric(df['dwc:decimalLongitude'], errors='coerce')
        else:
            df['latitude']  = np.nan
            df['longitude'] = np.nan
        return df

    df_all = extract_coords(df_all)
    df_coords = df_all.dropna(subset=['latitude', 'longitude']).copy()
    print(f'Records with coordinates: {len(df_coords)}')
    print(f'Records WITHOUT coordinates: {len(df_all) - len(df_coords)}')

    # Clip to Nevada bbox
    df_nv = df_coords[
        (df_coords['latitude']  >= NV_BBOX[1]) &
        (df_coords['latitude']  <= NV_BBOX[3]) &
        (df_coords['longitude'] >= NV_BBOX[0]) &
        (df_coords['longitude'] <= NV_BBOX[2])
    ].copy()
    print(f'Within Nevada bbox: {len(df_nv)}')
else:
    df_nv = pd.DataFrame()
    print('No iDigBio records retrieved.')


Total unique iDigBio records: 233
Columns present: ['basisofrecord', 'canonicalname', 'catalognumber', 'class', 'collectioncode', 'collectionid', 'collector', 'commonname', 'commonnames', 'continent', 'coordinateuncertainty', 'country', 'countrycode', 'county', 'datasetid', 'datecollected', 'datemodified', 'dc:language', 'dc:type', 'dcterms:accessRights', 'dcterms:bibliographicCitation', 'dcterms:language', 'dcterms:license', 'dcterms:modified', 'dcterms:references', 'dcterms:rightsHolder', 'dcterms:type', 'dqs', 'dwc:Identification', 'dwc:associatedMedia', 'dwc:associatedReferences', 'dwc:basisOfRecord', 'dwc:catalogNumber', 'dwc:class', 'dwc:collectionCode', 'dwc:collectionID', 'dwc:continent', 'dwc:coordinateUncertaintyInMeters', 'dwc:country', 'dwc:countryCode', 'dwc:county', 'dwc:dataGeneralizations', 'dwc:datasetID', 'dwc:datasetName', 'dwc:dateIdentified', 'dwc:day', 'dwc:decimalLatitude', 'dwc:decimalLongitude', 'dwc:degreeOfEstablishment', 'dwc:disposition', 'dwc:dynamicProper

In [8]:
## 8. Coordinate Precision Assessment

if len(df_nv):
    # coordinateuncertaintyinmeters may come from indexTerms or raw DwC
    unc_col = next((c for c in df_nv.columns
                    if 'coordinateuncertainty' in c.lower()), None)
    if unc_col:
        df_nv['coord_unc_m'] = pd.to_numeric(df_nv[unc_col], errors='coerce')
    else:
        df_nv['coord_unc_m'] = np.nan

    print('=== Coordinate Uncertainty Distribution ===')
    unc = df_nv['coord_unc_m'].dropna()
    if len(unc):
        print(f'  Records with uncertainty value: {len(unc)}')
        print(f'  Median uncertainty:   {unc.median():.0f}m')
        print(f'  <100m  (GPS-grade):   {(unc <  100).sum()}')
        print(f'  <1000m (field-grade): {(unc < 1000).sum()}')
        print(f'  <5000m (usable):      {(unc < 5000).sum()}')
        print(f'  >=5000m (too coarse): {(unc >= 5000).sum()}')
    else:
        print('  No uncertainty values populated — precision unknown.')

    print()
    # Find the best available name for institution, taxon, formation
    inst_col   = next((c for c in df_nv.columns if 'institutioncode' in c.lower()), None)
    name_col   = next((c for c in df_nv.columns if 'scientificname'  in c.lower()), None)
    form_col   = next((c for c in df_nv.columns if 'formation'       in c.lower()), None)

    if inst_col:
        print('=== Institution Breakdown ===')
        print(df_nv[inst_col].value_counts().to_string())
        print()
    if name_col:
        print('=== Scientific Names ===')
        print(df_nv[name_col].value_counts().to_string())
        print()
    if form_col:
        print('=== Formation Field ===')
        print(df_nv[form_col].value_counts().to_string())


=== Coordinate Uncertainty Distribution ===
  Records with uncertainty value: 15
  Median uncertainty:   473748m
  <100m  (GPS-grade):   0
  <1000m (field-grade): 0
  <5000m (usable):      0
  >=5000m (too coarse): 15

=== Institution Breakdown ===
institutioncode
cmc    15

=== Scientific Names ===
scientificname
cymbospondylus    15

=== Formation Field ===
formation
favret formation    15


In [9]:
## 9. Deduplicate Against Existing PBDB Records

if len(df_nv):
    df_nv['lon_r'] = df_nv['longitude'].round(2)
    df_nv['lat_r'] = df_nv['latitude'].round(2)
    df_nv['coord_pair'] = list(zip(df_nv['lon_r'], df_nv['lat_r']))

    df_new = df_nv[~df_nv['coord_pair'].isin(pbdb_coords)].copy()
    df_dup = df_nv[ df_nv['coord_pair'].isin(pbdb_coords)].copy()

    print(f'iDigBio records in Nevada bbox:       {len(df_nv)}')
    print(f'  Duplicates of existing PBDB coords: {len(df_dup)}')
    print(f'  NEW unique coordinates:             {len(df_new)}')
    print()

    if len(df_new):
        print('=== NEW LOCALITIES (not in PBDB) ===')
        name_col = next((c for c in df_new.columns if 'scientificname' in c.lower()), 'uuid')
        inst_col = next((c for c in df_new.columns if 'institutioncode' in c.lower()), None)
        form_col = next((c for c in df_new.columns if 'formation' in c.lower()), None)
        cat_col  = next((c for c in df_new.columns if 'catalognumber'  in c.lower()), None)
        show = ['latitude', 'longitude', 'coord_unc_m', name_col]
        for c in [form_col, inst_col, cat_col]:
            if c: show.append(c)
        print(df_new[show].to_string(index=False))
    else:
        print('No new coordinates beyond existing PBDB dataset.')
else:
    print('No Nevada iDigBio data to deduplicate.')


iDigBio records in Nevada bbox:       15
  Duplicates of existing PBDB coords: 0
  NEW unique coordinates:             15

=== NEW LOCALITIES (not in PBDB) ===
 latitude   longitude  coord_unc_m scientificname        formation institutioncode catalognumber
40.440358 -118.404438      96848.0 cymbospondylus favret formation             cmc        vp6399
40.440358 -118.404438      96848.0 cymbospondylus favret formation             cmc        vp6396
40.440358 -118.404438      96848.0 cymbospondylus favret formation             cmc       vp7396a
40.440358 -118.404438      96848.0 cymbospondylus favret formation             cmc       vp13158
38.661596 -116.866352     473748.0 cymbospondylus favret formation             cmc        vp9508
38.661596 -116.866352     473748.0 cymbospondylus favret formation             cmc       vp11831
38.661596 -116.866352     473748.0 cymbospondylus favret formation             cmc        vp9509
38.661596 -116.866352     473748.0 cymbospondylus favret formati

In [10]:
## 10. Save Outputs

if len(df_nv):
    df_nv.to_csv(PBDB_DIR / 'idigbio_nevada_triassic.csv', index=False)
    print(f'Saved: idigbio_nevada_triassic.csv ({len(df_nv)} records)')

    if len(df_new):
        df_new.to_csv(PBDB_DIR / 'idigbio_new_localities.csv', index=False)
        print(f'Saved: idigbio_new_localities.csv ({len(df_new)} new records)')

# Build combined dataset if new usable records exist
if len(df_nv) and len(df_new):
    name_col = next((c for c in df_new.columns if 'scientificname' in c.lower()), None)
    inst_col = next((c for c in df_new.columns if 'institutioncode' in c.lower()), None)
    form_col = next((c for c in df_new.columns if c == 'formation'), None)

    keep = ['uuid', 'latitude', 'longitude', 'coord_unc_m']
    for c in [name_col, inst_col, form_col]:
        if c and c not in keep: keep.append(c)

    df_idb_std = df_new[[c for c in keep if c in df_new.columns]].copy()
    df_idb_std = df_idb_std.rename(columns={
        'uuid':     'occurrence_id',
        name_col:   'taxon_name',
        inst_col:   'institutioncode',
    } if name_col else {})
    df_idb_std['source'] = 'idigbio'
    df_idb_std['order']  = 'Ichthyosauria'
    df_idb_std['in_nevada'] = True

    pbdb_std = pbdb[['occurrence_id','taxon_name','latitude','longitude','formation']].copy()
    pbdb_std['source'] = 'pbdb'
    pbdb_std['coord_unc_m'] = np.nan

    df_combined = pd.concat([pbdb_std, df_idb_std], ignore_index=True)
    df_combined.to_csv(PBDB_DIR / 'paleowave_combined.csv', index=False)
    print(f'Saved: paleowave_combined.csv ({len(df_combined)} total records)')
    print(f'  PBDB:    {(df_combined.source=="pbdb").sum()}')
    print(f'  iDigBio: {(df_combined.source=="idigbio").sum()}')

print()
print('=== VERDICT ===')
if len(df_nv) == 0:
    print('  >> iDigBio returned no Nevada ichthyosaur records.')
    print('  >> Recommend: check LACM, USNM, UNR museum portals directly.')
elif len(df_new) == 0:
    print(f'  >> iDigBio found {len(df_nv)} Nevada records but all duplicate PBDB coords.')
    print('  >> No new unique localities. PBDB is already the primary source.')
else:
    n_usable = (df_new['coord_unc_m'].fillna(9999) < 5000).sum()
    print(f'  >> {len(df_new)} new localities, {n_usable} with coord uncertainty <5km')
    if n_usable >= 5:
        print('  >> PROCEED: enough new localities to justify Phase 3 combined model')
    elif n_usable > 0:
        print('  >> MARGINAL: some new localities but small gain - evaluate case by case')
    else:
        print('  >> SKIP: new localities exist but coordinate precision too coarse to use')


Saved: idigbio_nevada_triassic.csv (15 records)
Saved: idigbio_new_localities.csv (15 new records)
Saved: paleowave_combined.csv (45 total records)
  PBDB:    30
  iDigBio: 15

=== VERDICT ===
  >> 15 new localities, 0 with coord uncertainty <5km
  >> SKIP: new localities exist but coordinate precision too coarse to use


In [11]:
## 11. Diagnostic — Inspect the 15 New Records

new = pd.read_csv(PBDB_DIR / 'idigbio_new_localities.csv')
print(f'=== 15 NEW iDigBio Records - Full Diagnostic ===')
print(f'Columns: {new.columns.tolist()}')
print()

# Coord uncertainty breakdown
unc_col = next((c for c in new.columns if 'coordinateuncertainty' in c.lower()), None)
if unc_col:
    print(f'=== Coordinate Uncertainty ({unc_col}) ===')
    print(new[[unc_col]].value_counts(dropna=False).to_string())
    print()

# Core fields
print('=== Coordinates + Taxon + Institution ===')
name_col = next((c for c in new.columns if 'scientificname' in c.lower()), None)
inst_col = next((c for c in new.columns if 'institutioncode' in c.lower()), None)
form_col = next((c for c in new.columns if 'formation' in c.lower()), None)
cat_col  = next((c for c in new.columns if 'catalognumber'  in c.lower()), None)
loc_col  = next((c for c in new.columns if 'locality'       in c.lower()
                 and 'geo' not in c.lower()), None)

show = ['latitude', 'longitude']
if unc_col:  show.append(unc_col)
for c in [name_col, form_col, inst_col, cat_col, loc_col]:
    if c: show.append(c)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 40)
print(new[show].to_string(index=False))
print()

# Are these Berlin-Ichthyosaur State Park?
# BISP coords: ~38.86N, -117.59W — wait that's Luning area
# Berlin-Ichthyosaur SP: ~38.93N, -117.63W
print('=== Distance from Berlin-Ichthyosaur State Park (38.93N, -117.63W) ===')
bisp_lat, bisp_lon = 38.93, -117.63
new['dist_bisp_km'] = (((new['latitude'] - bisp_lat)**2 +
                        (new['longitude'] - bisp_lon)**2)**0.5) * 111
print(new[['latitude','longitude','dist_bisp_km'] +
          ([name_col] if name_col else []) +
          ([inst_col] if inst_col else [])].to_string(index=False))
print()

# Check if any are genuinely outside the PBDB cluster (>50km from any PBDB point)
pbdb_orig = pd.read_csv(PBDB_DIR / 'pbdb_occurrences_clean.csv')
def min_dist_to_pbdb(row):
    dists = (((pbdb_orig['latitude'] - row['latitude'])**2 +
               (pbdb_orig['longitude'] - row['longitude'])**2)**0.5) * 111
    return dists.min()

new['min_dist_pbdb_km'] = new.apply(min_dist_to_pbdb, axis=1)
print('=== Minimum distance to any PBDB point ===')
print(new[['latitude','longitude','min_dist_pbdb_km','dist_bisp_km']].to_string(index=False))
print()
print(f'Records >50km from all PBDB points: {(new.min_dist_pbdb_km > 50).sum()}')
print(f'Records >20km from all PBDB points: {(new.min_dist_pbdb_km > 20).sum()}')
print(f'Records <10km from a PBDB point:    {(new.min_dist_pbdb_km < 10).sum()}')
print()
print('=== INTERPRETATION ===')
print('Records within 10km of PBDB: likely same site, centroid-shifted or institution offset')
print('Records >50km from PBDB:     potentially new geographic localities worth investigating')


=== 15 NEW iDigBio Records - Full Diagnostic ===
Columns: ['uuid', 'geopoint', 'taxonomicstatus', 'recordset', 'dqs', 'stateprovince', 'earliestepochorlowestseries', 'earliestperiodorlowestsystem', 'catalognumber', 'startdayofyear', 'collector', 'verbatimlocality', 'datemodified', 'countrycode', 'basisofrecord', 'institutioncode', 'mediarecords', 'continent', 'datecollected', 'etag', 'recordnumber', 'hasImage', 'highertaxon', 'collectionid', 'indexData', 'hasMedia', 'coordinateuncertainty', 'occurrenceid', 'earliestageorloweststage', 'institutionid', 'country', 'locality', 'collectioncode', 'canonicalname', 'eventdate', 'flags', 'verbatimeventdate', 'formation', 'recordids', 'datasetid', 'phylum', 'genus', 'scientificname', 'taxonrank', 'family', 'kingdom', 'taxonid', 'dwc:identificationRemarks', 'dc:language', 'dwc:verbatimCoordinates', 'dc:type', 'dwc:recordedBy', 'dwc:georeferencedDate', 'dcterms:accessRights', 'dwc:occurrenceID', 'dwc:dateIdentified', 'dwc:earliestEpochOrLowestSeri

## 12. Assessment & Recommendations

### What iDigBio returned

**15 records, 2 unique coordinate clusters, all *Cymbospondylus*, all CMC (Cincinnati Museum Center):**

| Cluster | Coords | Uncertainty | Count | Dist PBDB | Assessment |
|:--------|:-------|:-----------:|:-----:|:---------:|:-----------|
| A | 40.4404N 118.4044W | 96,848m (~97km) | 4 | 30km | Favret Formation centroid — not field GPS |
| B | 38.6616N 116.8664W | 473,748m (~474km) | 11 | 84km | State-level centroid — unusable |

### Verdict

**iDigBio does not improve PaleoWave training data in its current form.** Both coordinate clusters
are georeferenced to formation or state centroids, not actual collection localities. The 474km
uncertainty on Cluster B is essentially "somewhere in Nevada." Cluster A's 97km uncertainty
is consistent with a Favret Formation polygon centroid.

The CMC specimens (vp6396, vp6399, vp7396a, vp13158, vp9508–vp11832) are real physical
specimens — they just haven't been precisely georeferenced in iDigBio. The actual
collection locality data likely exists in the CMC's internal records.

### Phase 3 Recommendations

1. **Contact CMC directly** — Cincinnati Museum Center VP collection. Catalog numbers
   vp6396–vp13158 are Cymbospondylus from Favret Formation. Request precise locality
   data. Even formation + member + measured section reference would help.

2. **Check LACM and USNM** — LA County Museum of Natural History and Smithsonian
   National Museum of Natural History both have Nevada Triassic ichthyosaur material
   that may not be in iDigBio at all yet.

3. **UNR and Nevada State Museum** — University of Nevada Reno and the Nevada State
   Museum (Carson City) hold locally-collected material with better georeferencing
   than out-of-state institutions. Neither is well-represented in iDigBio.

4. **PBDB literature harvest** — Several Triassic ichthyosaur papers (Camp 1980,
   Nicholls & Manabe 2004, Sander et al. 2011) include locality coordinates in
   the text that were never entered into PBDB. A targeted literature harvest
   could add 5–10 Luning/Gabbs formation localities.

5. **Proceed with Phase 3 on current 18 localities** — The formation-stratified
   model (Prida/Favret north vs Luning/Gabbs south) is the highest-leverage
   improvement available with existing data. Don't wait for iDigBio to improve.
